In [1]:
import pandas as pd
import numpy as np

file_path = "../data/raw/CanAI Cafe 2023 Sales Information.csv"

df = pd.read_csv(file_path)

df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Province
0,TXN_9687814,coffee,2,3.5,7.0,Digital Wallet,In-store,2023-09-28,British Columbia
1,TXN_7002925,Refresher,2,5.0,10.0,Cash,NaN,2023-05-02,NL
2,TXN_7668262,Donut,1,2.0,2.0,Digital Wallet,In-store,2023-11-27,British Columbia
3,TXN_8322624,Refresher,3,5.0,15.0,ERR_PM_102,In-store,NaN,Saskatchewan
4,TXN_9390285,Coffee,1,3.5,3.5,ERR_PM_102,Takeaway,2023-05-11,newfoundland and labrador


## Missing and Blank Value Analysis

This section identifies columns with missing values, including both standard missing values and blank string entries.

In [2]:
# Standard missing values
standard_missing = df.isna().sum()

# Blank string values
blank_missing = df.apply(
    lambda col: col.astype(str).str.strip().eq("").sum()
)

# Total missing = NaN + blank strings
total_missing = standard_missing + blank_missing

missing_report = pd.DataFrame({
    "standard_missing": standard_missing,
    "blank_entries": blank_missing,
    "total_missing": total_missing,
    "total_missing_percent": ((total_missing / len(df)) * 100).round(2)
}).sort_values(by="total_missing", ascending=False)

missing_report

,standard_missing,blank_entries,total_missing,total_missing_percent
Location,715,163,878,8.78
Payment Method,432,95,527,5.27
Province,350,67,417,4.17
Transaction Date,176,39,215,2.15
Item,87,13,100,1.00
Quantity,78,20,98,0.98
Transaction ID,0,0,0,0.00
Total Spent,0,0,0,0.00
Price Per Unit,0,0,0,0.00


In [4]:
df = df.replace(r"^\s*$", np.nan, regex=True)

df.isna().sum().sort_values(ascending=False)

Location            878
Payment Method      527
Province            417
Transaction Date    215
Item                100
Quantity             98
Transaction ID        0
Total Spent           0
Price Per Unit        0
dtype: int64

In [5]:
# =========================
# Drop rows with missing values in selected columns
# =========================

df_clean = df.copy()

# Convert blank entries to NaN
df_clean = df_clean.replace(r"^\s*$", np.nan, regex=True)

columns_to_drop_missing_rows = [
    "Province",
    "Transaction Date",
    "Item"
]

rows_before = len(df_clean)

df_clean = df_clean.dropna(subset=columns_to_drop_missing_rows)

rows_after = len(df_clean)
rows_removed = rows_before - rows_after

print(f"Rows before: {rows_before}")
print(f"Rows after: {rows_after}")
print(f"Rows removed: {rows_removed}")
print(f"Percentage removed: {(rows_removed / rows_before) * 100:.2f}%")

# =========================
# Missing Report After Dropping Rows
# =========================

missing_report_after = pd.DataFrame({
    "missing_count": df_clean.isna().sum(),
    "missing_percent": (df_clean.isna().mean() * 100).round(2)
}).sort_values(by="missing_percent", ascending=False)

display(missing_report_after)

Rows before: 10000
Rows after: 9284
Rows removed: 716
Percentage removed: 7.16%


,missing_count,missing_percent
Location,812,8.75
Payment Method,489,5.27
Quantity,89,0.96
Transaction ID,0,0.00
Item,0,0.00
Total Spent,0,0.00
Price Per Unit,0,0.00
Transaction Date,0,0.00
Province,0,0.00


In [6]:
import pandas as pd
import numpy as np

df_missing_analysis = df.copy()

# Convert blank entries to proper NaN
df_missing_analysis = df_missing_analysis.replace(r"^\s*$", np.nan, regex=True)

# Create a flag for rows where Province is missing
df_missing_analysis["Province Missing"] = df_missing_analysis["Province"].isna()

# Overall Province missing rate
province_missing_count = df_missing_analysis["Province Missing"].sum()
province_missing_percent = df_missing_analysis["Province Missing"].mean() * 100

print(f"Rows with missing Province: {province_missing_count}")
print(f"Province missing percentage: {province_missing_percent:.2f}%")

Rows with missing Province: 417
Province missing percentage: 4.17%


In [7]:
df_missing_analysis["Province Missing"].value_counts(dropna=False)
df_missing_analysis["Province Missing"].value_counts(normalize=True) * 100

Province Missing
False    95.83
True      4.17
Name: proportion, dtype: float64

In [8]:
province_missing = df_missing_analysis["Province Missing"]

missing_when_province_missing = (
    df_missing_analysis[province_missing]
    .isna()
    .mean() * 100
)

missing_when_province_present = (
    df_missing_analysis[~province_missing]
    .isna()
    .mean() * 100
)

co_missing_report = pd.DataFrame({
    "missing_percent_when_province_missing": missing_when_province_missing.round(2),
    "missing_percent_when_province_present": missing_when_province_present.round(2),
})

co_missing_report["difference_percentage_points"] = (
    co_missing_report["missing_percent_when_province_missing"] -
    co_missing_report["missing_percent_when_province_present"]
).round(2)

co_missing_report = co_missing_report.sort_values(
    by="difference_percentage_points",
    ascending=False
)

display(co_missing_report)

,missing_percent_when_province_missing,missing_percent_when_province_present,difference_percentage_points
Province,100.00,0.00,100.00
Quantity,1.92,0.94,0.98
Location,9.11,8.77,0.34
Item,1.20,0.99,0.21
Payment Method,5.28,5.27,0.01
Transaction ID,0.00,0.00,0.00
Total Spent,0.00,0.00,0.00
Price Per Unit,0.00,0.00,0.00
Province Missing,0.00,0.00,0.00
Transaction Date,1.92,2.16,-0.24


In [9]:
payment_method_province_missing = (
    df_missing_analysis
    .groupby("Payment Method", dropna=False)["Province Missing"]
    .agg(
        total_rows="count",
        missing_province_count="sum",
        missing_province_percent="mean"
    )
    .reset_index()
)

payment_method_province_missing["missing_province_percent"] = (
    payment_method_province_missing["missing_province_percent"] * 100
).round(2)

payment_method_province_missing = payment_method_province_missing.sort_values(
    by="missing_province_percent",
    ascending=False
)

display(payment_method_province_missing)

,Payment Method,total_rows,missing_province_count,missing_province_percent
1,Credit Card,3885,171,4.40
4,NaN,527,22,4.17
2,Digital Wallet,3001,123,4.10
0,Cash,2559,100,3.91
3,ERR_PM_102,28,1,3.57


In [10]:
item_province_missing = (
    df_missing_analysis
    .groupby("Item", dropna=False)["Province Missing"]
    .agg(
        total_rows="count",
        missing_province_count="sum",
        missing_province_percent="mean"
    )
    .reset_index()
)

item_province_missing["missing_province_percent"] = (
    item_province_missing["missing_province_percent"] * 100
).round(2)

# Only show items with at least 10 rows to avoid misleading tiny groups
item_province_missing_filtered = item_province_missing[
    item_province_missing["total_rows"] >= 10
].sort_values(by="missing_province_percent", ascending=False)

display(item_province_missing_filtered)

,Item,total_rows,missing_province_count,missing_province_percent
23,sandwich,111,6,5.41
18,cofee,112,6,5.36
25,NaN,100,5,5.00
15,Tea,1257,62,4.93
6,Doughnut,64,3,4.69
4,Donut,936,43,4.59
19,coffe,112,5,4.46
13,Sandwich,1060,47,4.43
2,Coffee,2248,99,4.40
11,Salad,748,30,4.01


In [11]:
df_missing_analysis["Transaction Date"] = pd.to_datetime(
    df_missing_analysis["Transaction Date"],
    errors="coerce"
)

df_missing_analysis["Transaction Month"] = (
    df_missing_analysis["Transaction Date"]
    .dt.to_period("M")
)

province_missing_by_month = (
    df_missing_analysis
    .groupby("Transaction Month", dropna=False)["Province Missing"]
    .agg(
        total_rows="count",
        missing_province_count="sum",
        missing_province_percent="mean"
    )
    .reset_index()
)

province_missing_by_month["missing_province_percent"] = (
    province_missing_by_month["missing_province_percent"] * 100
).round(2)

display(province_missing_by_month)

,Transaction Month,total_rows,missing_province_count,missing_province_percent
0,2023-01,851,33,3.88
1,2023-02,765,31,4.05
2,2023-03,839,44,5.24
3,2023-04,766,26,3.39
4,2023-05,822,31,3.77
5,2023-06,848,42,4.95
6,2023-07,746,24,3.22
7,2023-08,871,33,3.79
8,2023-09,782,30,3.84
9,2023-10,839,38,4.53


In [12]:
import pandas as pd
import numpy as np

# Work on a copy so we do not accidentally change the original dataset
df_missing_analysis = df.copy()

# Convert blank strings like "", " ", "   " into NaN
df_missing_analysis = df_missing_analysis.replace(r"^\s*$", np.nan, regex=True)

# Columns we want to investigate
columns_to_investigate = [
    "Payment Method",
    "Location",
    "Transaction Date",
    "Item",
    "Quantity"
]

# Keep only columns that actually exist in the dataset
columns_to_investigate = [
    col for col in columns_to_investigate
    if col in df_missing_analysis.columns
]

print("Columns being investigated:")
print(columns_to_investigate)

# Create missing flags for each column
for col in columns_to_investigate:
    df_missing_analysis[f"{col} Missing"] = df_missing_analysis[col].isna()

Columns being investigated:
['Payment Method', 'Location', 'Transaction Date', 'Item', 'Quantity']


In [13]:
missing_investigation_report = pd.DataFrame({
    "missing_count": df_missing_analysis[columns_to_investigate].isna().sum(),
    "missing_percent": (
        df_missing_analysis[columns_to_investigate].isna().mean() * 100
    ).round(2)
}).sort_values(by="missing_percent", ascending=False)

display(missing_investigation_report)

,missing_count,missing_percent
Location,878,8.78
Payment Method,527,5.27
Transaction Date,215,2.15
Item,100,1.00
Quantity,98,0.98


In [14]:
# Pairwise co-missingness report
co_missing_matrix = pd.DataFrame(
    index=columns_to_investigate,
    columns=columns_to_investigate
)

for target_col in columns_to_investigate:
    target_missing = df_missing_analysis[target_col].isna()
    
    for other_col in columns_to_investigate:
        if target_missing.sum() == 0:
            co_missing_matrix.loc[target_col, other_col] = 0
        else:
            percent_missing_together = (
                df_missing_analysis.loc[target_missing, other_col].isna().mean() * 100
            )
            co_missing_matrix.loc[target_col, other_col] = round(percent_missing_together, 2)

display(co_missing_matrix)

,Payment Method,Location,Transaction Date,Item,Quantity
Payment Method,100.0,15.75,1.9,1.14,0.76
Location,9.45,100.0,2.39,0.91,0.68
Transaction Date,4.65,9.77,100.0,1.4,0.47
Item,6.0,8.0,3.0,100.0,0.0
Quantity,4.08,6.12,1.02,0.0,100.0


In [15]:
for target_col in columns_to_investigate:
    print(f"\n==============================")
    print(f"Missingness relationship for: {target_col}")
    print(f"==============================")
    
    target_missing = df_missing_analysis[target_col].isna()
    
    missing_when_target_missing = (
        df_missing_analysis[target_missing]
        .isna()
        .mean() * 100
    )
    
    missing_when_target_present = (
        df_missing_analysis[~target_missing]
        .isna()
        .mean() * 100
    )
    
    report = pd.DataFrame({
        f"missing_percent_when_{target_col}_missing": missing_when_target_missing.round(2),
        f"missing_percent_when_{target_col}_present": missing_when_target_present.round(2),
    })
    
    report["difference_percentage_points"] = (
        report[f"missing_percent_when_{target_col}_missing"] -
        report[f"missing_percent_when_{target_col}_present"]
    ).round(2)
    
    report = report.sort_values(
        by="difference_percentage_points",
        ascending=False
    )
    
    display(report)


Missingness relationship for: Payment Method


,missing_percent_when_Payment Method_missing,missing_percent_when_Payment Method_present,difference_percentage_points
Payment Method,100.00,0.00,100.00
Location,15.75,8.39,7.36
Item,1.14,0.99,0.15
Transaction ID,0.00,0.00,0.00
Total Spent,0.00,0.00,0.00
Price Per Unit,0.00,0.00,0.00
Item Missing,0.00,0.00,0.00
Province,4.17,4.17,0.00
Payment Method Missing,0.00,0.00,0.00
Location Missing,0.00,0.00,0.00



Missingness relationship for: Location


,missing_percent_when_Location_missing,missing_percent_when_Location_present,difference_percentage_points
Location,100.00,0.00,100.00
Payment Method,9.45,4.87,4.58
Transaction Date,2.39,2.13,0.26
Province,4.33,4.15,0.18
Total Spent,0.00,0.00,0.00
Transaction ID,0.00,0.00,0.00
Location Missing,0.00,0.00,0.00
Transaction Date Missing,0.00,0.00,0.00
Price Per Unit,0.00,0.00,0.00
Payment Method Missing,0.00,0.00,0.00



Missingness relationship for: Transaction Date


,missing_percent_when_Transaction Date_missing,missing_percent_when_Transaction Date_present,difference_percentage_points
Transaction Date,100.00,0.00,100.00
Location,9.77,8.76,1.01
Item,1.40,0.99,0.41
Total Spent,0.00,0.00,0.00
Price Per Unit,0.00,0.00,0.00
Transaction ID,0.00,0.00,0.00
Item Missing,0.00,0.00,0.00
Transaction Date Missing,0.00,0.00,0.00
Payment Method Missing,0.00,0.00,0.00
Location Missing,0.00,0.00,0.00



Missingness relationship for: Item


,missing_percent_when_Item_missing,missing_percent_when_Item_present,difference_percentage_points
Item,100.0,0.00,100.00
Transaction Date,3.0,2.14,0.86
Province,5.0,4.16,0.84
Payment Method,6.0,5.26,0.74
Transaction ID,0.0,0.00,0.00
Price Per Unit,0.0,0.00,0.00
Item Missing,0.0,0.00,0.00
Total Spent,0.0,0.00,0.00
Payment Method Missing,0.0,0.00,0.00
Location Missing,0.0,0.00,0.00



Missingness relationship for: Quantity


,missing_percent_when_Quantity_missing,missing_percent_when_Quantity_present,difference_percentage_points
Quantity,100.00,0.00,100.00
Province,8.16,4.13,4.03
Total Spent,0.00,0.00,0.00
Price Per Unit,0.00,0.00,0.00
Payment Method Missing,0.00,0.00,0.00
Transaction ID,0.00,0.00,0.00
Item Missing,0.00,0.00,0.00
Quantity Missing,0.00,0.00,0.00
Transaction Date Missing,0.00,0.00,0.00
Location Missing,0.00,0.00,0.00


In [16]:
categorical_columns_to_compare = [
    "Payment Method",
    "Location",
    "Item"
]

categorical_columns_to_compare = [
    col for col in categorical_columns_to_compare
    if col in df_missing_analysis.columns
]

minimum_group_size = 10

for target_col in columns_to_investigate:
    target_missing_flag = f"{target_col} Missing"
    
    print(f"\n########################################")
    print(f"Categorical relationship checks for missing {target_col}")
    print(f"########################################")
    
    for category_col in categorical_columns_to_compare:
        
        # Avoid grouping a column by itself
        if category_col == target_col:
            continue
        
        print(f"\nMissing {target_col} by {category_col}")
        
        report = (
            df_missing_analysis
            .groupby(category_col, dropna=False)[target_missing_flag]
            .agg(
                total_rows="count",
                missing_count="sum",
                missing_percent="mean"
            )
            .reset_index()
        )
        
        report["missing_percent"] = (
            report["missing_percent"] * 100
        ).round(2)
        
        # Avoid misleading results from tiny groups
        report = report[report["total_rows"] >= minimum_group_size]
        
        report = report.sort_values(
            by="missing_percent",
            ascending=False
        )
        
        display(report)


########################################
Categorical relationship checks for missing Payment Method
########################################

Missing Payment Method by Location


,Location,total_rows,missing_count,missing_percent
2,NaN,878,83,9.45
1,Takeaway,3207,173,5.39
0,In-store,5915,271,4.58



Missing Payment Method by Item


,Item,total_rows,missing_count,missing_percent
23,sandwich,111,14,12.61
19,coffe,112,10,8.93
14,TEA,63,5,7.94
5,Donutt,56,4,7.14
17,Tee,78,5,6.41
18,cofee,112,7,6.25
25,NaN,100,6,6.00
15,Tea,1257,74,5.89
12,Sandwhich,119,7,5.88
9,Juicee,35,2,5.71



########################################
Categorical relationship checks for missing Location
########################################

Missing Location by Payment Method


...

,Payment Method,total_rows,missing_count,missing_percent
3,ERR_PM_102,28,1,3.57
0,Cash,2559,29,1.13
2,Digital Wallet,3001,30,1.00
1,Credit Card,3885,34,0.88
4,NaN,527,4,0.76



Missing Quantity by Location


,Location,total_rows,missing_count,missing_percent
1,Takeaway,3207,36,1.12
0,In-store,5915,56,0.95
2,NaN,878,6,0.68



Missing Quantity by Item


,Item,total_rows,missing_count,missing_percent
21,donut,67,2,2.99
16,Tea,77,2,2.60
12,Sandwhich,119,2,1.68
6,Doughnut,64,1,1.56
3,Cookie,960,14,1.46
17,Tee,78,1,1.28
15,Tea,1257,15,1.19
4,Donut,936,11,1.18
13,Sandwich,1060,11,1.04
0,C0ffee,97,1,1.03


In [17]:
if "Transaction Date" in df_missing_analysis.columns:
    df_missing_analysis["Transaction Date Parsed"] = pd.to_datetime(
        df_missing_analysis["Transaction Date"],
        errors="coerce"
    )
    
    df_missing_analysis["Transaction Month"] = (
        df_missing_analysis["Transaction Date Parsed"]
        .dt.to_period("M")
    )
    
    for target_col in columns_to_investigate:
        target_missing_flag = f"{target_col} Missing"
        
        print(f"\nMissing {target_col} by Transaction Month")
        
        monthly_report = (
            df_missing_analysis
            .groupby("Transaction Month", dropna=False)[target_missing_flag]
            .agg(
                total_rows="count",
                missing_count="sum",
                missing_percent="mean"
            )
            .reset_index()
        )
        
        monthly_report["missing_percent"] = (
            monthly_report["missing_percent"] * 100
        ).round(2)
        
        display(monthly_report)


Missing Payment Method by Transaction Month


,Transaction Month,total_rows,missing_count,missing_percent
0,2023-01,851,45,5.29
1,2023-02,765,36,4.71
2,2023-03,839,51,6.08
3,2023-04,766,42,5.48
4,2023-05,822,43,5.23
5,2023-06,848,42,4.95
6,2023-07,746,29,3.89
7,2023-08,871,61,7.00
8,2023-09,782,38,4.86
9,2023-10,839,51,6.08



Missing Location by Transaction Month


,Transaction Month,total_rows,missing_count,missing_percent
0,2023-01,851,75,8.81
1,2023-02,765,70,9.15
2,2023-03,839,80,9.54
3,2023-04,766,65,8.49
4,2023-05,822,68,8.27
5,2023-06,848,59,6.96
6,2023-07,746,70,9.38
7,2023-08,871,67,7.69
8,2023-09,782,62,7.93
9,2023-10,839,81,9.65



Missing Transaction Date by Transaction Month


,Transaction Month,total_rows,missing_count,missing_percent
0,2023-01,851,0,0.00
1,2023-02,765,0,0.00
2,2023-03,839,0,0.00
3,2023-04,766,0,0.00
4,2023-05,822,0,0.00
5,2023-06,848,0,0.00
6,2023-07,746,0,0.00
7,2023-08,871,0,0.00
8,2023-09,782,0,0.00
9,2023-10,839,0,0.00



Missing Item by Transaction Month


,Transaction Month,total_rows,missing_count,missing_percent
0,2023-01,851,13,1.53
1,2023-02,765,5,0.65
2,2023-03,839,10,1.19
3,2023-04,766,6,0.78
4,2023-05,822,8,0.97
5,2023-06,848,9,1.06
6,2023-07,746,9,1.21
7,2023-08,871,5,0.57
8,2023-09,782,10,1.28
9,2023-10,839,8,0.95



Missing Quantity by Transaction Month


,Transaction Month,total_rows,missing_count,missing_percent
0,2023-01,851,9,1.06
1,2023-02,765,6,0.78
2,2023-03,839,8,0.95
3,2023-04,766,13,1.70
4,2023-05,822,6,0.73
5,2023-06,848,5,0.59
6,2023-07,746,7,0.94
7,2023-08,871,12,1.38
8,2023-09,782,10,1.28
9,2023-10,839,1,0.12


In [18]:
numeric_columns_to_compare = [
    "Quantity",
    "Price Per Unit",
    "Total Spent"
]

numeric_columns_to_compare = [
    col for col in numeric_columns_to_compare
    if col in df_missing_analysis.columns
]

# Convert numeric columns safely
for col in numeric_columns_to_compare:
    df_missing_analysis[col] = pd.to_numeric(
        df_missing_analysis[col],
        errors="coerce"
    )

for target_col in columns_to_investigate:
    target_missing_flag = f"{target_col} Missing"
    
    print(f"\nNumeric comparison for rows with missing {target_col}")
    
    numeric_comparison = (
        df_missing_analysis
        .groupby(target_missing_flag)[numeric_columns_to_compare]
        .agg(["count", "mean", "median", "min", "max"])
    )
    
    display(numeric_comparison)


Numeric comparison for rows with missing Payment Method


Quantity                            Price Per Unit  \
                          count      mean median  min  max          count   
Payment Method Missing                                                      
False                      9379  1.985073    2.0  1.0  5.0           9473   
True                        523  2.015296    2.0  1.0  5.0            527   

                                                  Total Spent            \
                            mean median  min  max       count      mean   
Payment Method Missing                                                    
False                   4.345878    3.5  2.0  9.0        9473  8.628154   
True                    4.326376    3.5  2.0  9.0         527  8.640417   

                                          
                       median  min   max  
Payment Method Missing                    
False                     7.0  2.0  45.0  
True                      7.0  2.0  45.0


Numeric comparison for rows with missing Location


Quantity                            Price Per Unit            \
                    count      mean median  min  max          count      mean   
Location Missing                                                                
False                9030  1.987929    2.0  1.0  5.0           9122  4.331342   
True                  872  1.973624    2.0  1.0  5.0            878  4.485194   

                                  Total Spent                              
                 median  min  max       count      mean median  min   max  
Location Missing                                                           
False               3.5  2.0  9.0        9122  8.609899    7.0  2.0  45.0  
True                3.5  2.0  9.0         878  8.825171    7.0  2.0  45.0


Numeric comparison for rows with missing Transaction Date


Quantity                            Price Per Unit  \
                            count      mean median  min  max          count   
Transaction Date Missing                                                      
False                        9688  1.988955    2.0  1.0  5.0           9785   
True                          214  1.883178    2.0  1.0  5.0            215   

                                                    Total Spent            \
                              mean median  min  max       count      mean   
Transaction Date Missing                                                    
False                     4.342667    3.5  2.0  9.0        9785  8.636382   
True                      4.444186    3.5  2.0  9.0         215  8.283721   

                                            
                         median  min   max  
Transaction Date Missing                    
False                       7.0  2.0  45.0  
True                        7.0  2.0  36.0


Numeric comparison for rows with missing Item


Quantity                            Price Per Unit            \
                count      mean median  min  max          count      mean   
Item Missing                                                                
False            9802  1.986125    2.0  1.0  5.0           9900  4.344242   
True              100  2.040000    2.0  1.0  5.0            100  4.405000   

                              Total Spent                              
             median  min  max       count      mean median  min   max  
Item Missing                                                           
False           3.5  2.0  9.0        9900  8.626313    7.0  2.0  45.0  
True            3.5  2.0  9.0         100  8.875000    7.0  2.0  32.0


Numeric comparison for rows with missing Quantity


Quantity                            Price Per Unit            \
                    count      mean median  min  max          count      mean   
Quantity Missing                                                                
False                9902  1.986669    2.0  1.0  5.0           9902  4.348212   
True                    0       NaN    NaN  NaN  NaN             98  4.005102   

                                  Total Spent                              
                 median  min  max       count      mean median  min   max  
Quantity Missing                                                           
False               3.5  2.0  9.0        9902  8.637497    7.0  2.0  45.0  
True                3.5  2.0  9.0          98  7.750000    5.0  2.0  40.0

In [19]:
for target_col in columns_to_investigate:
    print(f"\nRows where {target_col} is missing")
    
    rows_missing_target = df_missing_analysis[
        df_missing_analysis[target_col].isna()
    ]
    
    display(rows_missing_target.head(20))


Rows where Payment Method is missing


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Province,Payment Method Missing,Location Missing,Transaction Date Missing,Item Missing,Quantity Missing,Transaction Date Parsed,Transaction Month
33,TXN_5137655,Coffee,4.0,3.5,14.0,NaN,NaN,2023-05-09,Manitoba,True,True,False,False,False,2023-05-09,2023-05
80,TXN_3396634,Tea,3.0,3.0,9.0,NaN,In-store,2023-09-09,British Columbia,True,False,False,False,False,2023-09-09,2023-09
82,TXN_9358343,Coffee,3.0,3.5,10.5,NaN,In-store,2023-01-09,British Columbia,True,False,False,False,False,2023-01-09,2023-01
85,TXN_5219124,Tea,1.0,3.0,3.0,NaN,NaN,2023-08-15,New Foundland,True,True,False,False,False,2023-08-15,2023-08
120,TXN_1576243,sandwich,1.0,8.0,8.0,NaN,NaN,2023-05-04,Saskatchewan,True,True,False,False,False,2023-05-04,2023-05
128,TXN_5651592,Coffee,1.0,3.5,3.5,NaN,In-store,2023-09-29,British Columbia,True,False,False,False,False,2023-09-29,2023-09
157,TXN_6042672,Coffee,1.0,3.5,3.5,NaN,NaN,2023-12-08,Saskatchewn,True,True,False,False,False,2023-12-08,2023-12
170,TXN_5008062,Coffee,4.0,3.5,14.0,NaN,Takeaway,2023-10-10,British Columbia,True,False,False,False,False,2023-10-10,2023-10
206,TXN_8356426,Coffee,1.0,3.5,3.5,NaN,NaN,2023-12-01,Saskatchewan,True,True,False,False,False,2023-12-01,2023-12
217,TXN_2707119,Tea,1.0,3.0,3.0,NaN,In-store,2023-10-04,British Columbia,True,False,False,False,False,2023-10-04,2023-10



Rows where Location is missing


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Province,Payment Method Missing,Location Missing,Transaction Date Missing,Item Missing,Quantity Missing,Transaction Date Parsed,Transaction Month
1,TXN_7002925,Refresher,2.0,5.0,10.0,Cash,NaN,2023-05-02,NL,False,True,False,False,False,2023-05-02,2023-05
33,TXN_5137655,Coffee,4.0,3.5,14.0,NaN,NaN,2023-05-09,Manitoba,True,True,False,False,False,2023-05-09,2023-05
51,TXN_9671923,Donut,1.0,2.0,2.0,Credit Card,NaN,2023-07-09,Manitoba,False,True,False,False,False,2023-07-09,2023-07
55,TXN_9299675,Sandwich,3.0,8.0,24.0,Digital Wallet,NaN,2023-05-17,Newfoundland,False,True,False,False,False,2023-05-17,2023-05
72,TXN_4204740,Coffee,1.0,3.5,3.5,Credit Card,NaN,2023-07-20,NaN,False,True,False,False,False,2023-07-20,2023-07
73,TXN_1839747,Donut,2.0,2.0,4.0,Cash,NaN,2023-11-22,Newfoundland,False,True,False,False,False,2023-11-22,2023-11
85,TXN_5219124,Tea,1.0,3.0,3.0,NaN,NaN,2023-08-15,New Foundland,True,True,False,False,False,2023-08-15,2023-08
113,TXN_5385028,Coffee,3.0,3.5,10.5,Digital Wallet,NaN,2023-02-27,Newfoundland,False,True,False,False,False,2023-02-27,2023-02
120,TXN_1576243,sandwich,1.0,8.0,8.0,NaN,NaN,2023-05-04,Saskatchewan,True,True,False,False,False,2023-05-04,2023-05
123,TXN_2029245,Juice,1.0,4.5,4.5,Credit Card,NaN,2023-08-17,Saskatchewan,False,True,False,False,False,2023-08-17,2023-08



Rows where Transaction Date is missing


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Province,Payment Method Missing,Location Missing,Transaction Date Missing,Item Missing,Quantity Missing,Transaction Date Parsed,Transaction Month
3,TXN_8322624,Refresher,3.0,5.0,15.0,ERR_PM_102,In-store,NaN,Saskatchewan,False,False,True,False,False,NaT,NaT
21,TXN_7815208,Cookie,1.0,2.5,2.5,Digital Wallet,In-store,NaN,Newfoundland,False,False,True,False,False,NaT,NaT
62,TXN_7857240,Coffee,1.0,3.5,3.5,Credit Card,In-store,NaN,British Columbia,False,False,True,False,False,NaT,NaT
166,TXN_6157885,Sandwich,4.0,8.0,32.0,Credit Card,In-store,NaN,British Columbia,False,False,True,False,False,NaT,NaT
229,TXN_1713388,cofee,1.0,3.5,3.5,Cash,In-store,NaN,British Columbia,False,False,True,False,False,NaT,NaT
250,TXN_5243325,Donutt,1.0,2.0,2.0,Cash,In-store,NaN,Saskatchewan,False,False,True,False,False,NaT,NaT
285,TXN_9341004,Sandwich,2.0,8.0,16.0,Credit Card,In-store,NaN,B.C.,False,False,True,False,False,NaT,NaT
394,TXN_9847184,Refresher,1.0,5.0,5.0,Digital Wallet,NaN,NaN,British Columbia,False,True,True,False,False,NaT,NaT
531,TXN_1157013,Tea,2.0,3.0,6.0,Credit Card,NaN,NaN,British Columbia,False,True,True,False,False,NaT,NaT
543,TXN_9245447,Donut,5.0,2.0,10.0,Credit Card,In-store,NaN,Newfoundland,False,False,True,False,False,NaT,NaT



Rows where Item is missing


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Province,Payment Method Missing,Location Missing,Transaction Date Missing,Item Missing,Quantity Missing,Transaction Date Parsed,Transaction Month
110,TXN_5176763,NaN,2.0,4.5,9.0,Cash,In-store,2023-05-23,Sask.,False,False,False,True,False,2023-05-23,2023-05
278,TXN_3401905,NaN,1.0,3.5,3.5,Digital Wallet,Takeaway,2023-12-19,NaN,False,False,False,True,False,2023-12-19,2023-12
306,TXN_7906172,NaN,2.0,3.0,6.0,Credit Card,Takeaway,2023-06-07,Manitoba,False,False,False,True,False,2023-06-07,2023-06
365,TXN_6392674,NaN,3.0,2.0,6.0,Digital Wallet,In-store,2023-06-20,Sasktchewan,False,False,False,True,False,2023-06-20,2023-06
417,TXN_1742315,NaN,4.0,3.0,12.0,Cash,NaN,2023-03-15,Saskatchewan,False,True,False,True,False,2023-03-15,2023-03
423,TXN_7935078,NaN,2.0,8.0,16.0,Cash,Takeaway,2023-01-07,Newfoundland,False,False,False,True,False,2023-01-07,2023-01
424,TXN_2231990,NaN,5.0,3.0,15.0,NaN,NaN,2023-02-02,British Columbia,True,True,False,True,False,2023-02-02,2023-02
457,TXN_8462095,NaN,1.0,2.5,2.5,Credit Card,In-store,2023-12-25,British Columba,False,False,False,True,False,2023-12-25,2023-12
686,TXN_9434242,NaN,2.0,3.5,7.0,Credit Card,In-store,2023-09-16,Newfoundland,False,False,False,True,False,2023-09-16,2023-09
721,TXN_1135547,NaN,4.0,3.5,14.0,Credit Card,In-store,2023-01-26,Newfoundland,False,False,False,True,False,2023-01-26,2023-01



Rows where Quantity is missing


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Province,Payment Method Missing,Location Missing,Transaction Date Missing,Item Missing,Quantity Missing,Transaction Date Parsed,Transaction Month
38,TXN_3174000,sandwich,NaN,8.0,8.0,Credit Card,In-store,2023-03-03,Manitoba,False,False,False,False,True,2023-03-03,2023-03
91,TXN_4388592,Coffee,NaN,3.5,3.5,Digital Wallet,Takeaway,2023-05-15,Manitoba,False,False,False,False,True,2023-05-15,2023-05
290,TXN_1337498,Juice,NaN,4.5,9.0,Digital Wallet,In-store,2023-04-21,Saskatchewan,False,False,False,False,True,2023-04-21,2023-04
314,TXN_3492647,Donut,NaN,2.0,2.0,Cash,In-store,2023-08-17,Newfoundland,False,False,False,False,True,2023-08-17,2023-08
347,TXN_7876345,Coffee,NaN,3.5,7.0,Credit Card,In-store,2023-03-14,Newfoundland,False,False,False,False,True,2023-03-14,2023-03
390,TXN_7332287,Tee,NaN,3.0,3.0,Credit Card,In-store,2023-03-02,Saskatchewan,False,False,False,False,True,2023-03-02,2023-03
581,TXN_5535975,Donut,NaN,2.0,4.0,Digital Wallet,Takeaway,2023-05-24,Sask.,False,False,False,False,True,2023-05-24,2023-05
598,TXN_7089321,Donut,NaN,2.0,2.0,Cash,In-store,2023-01-25,British Columbia,False,False,False,False,True,2023-01-25,2023-01
742,TXN_2881591,Sandwich,NaN,8.0,24.0,Digital Wallet,Takeaway,2023-08-06,British Columbia,False,False,False,False,True,2023-08-06,2023-08
960,TXN_1151773,Coffee,NaN,3.5,14.0,Cash,In-store,2023-03-11,Manitoba,False,False,False,False,True,2023-03-11,2023-03
